In [ ]:
"""
Objective: Execute the complete Machine Learning pipeline of the study.
Includes Unsupervised Learning (K-Means for macro-regime detection) and 
Supervised Learning (XGBoost with SHAP Values for predictive interpretability).
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')

# ==========================================
# GENERAL PARAMETERS & LOADING
# ==========================================
BASE_FILE = 'clean_macro_fred_dataset.csv'
START_DATE, END_DATE = '2005-01-01', '2026-04-01'

print("[*] Loading clean dataset...")
df_full = pd.read_csv(BASE_FILE, index_col=0, parse_dates=True)
df_model = df_full.loc[START_DATE:END_DATE].copy()

# ==============================================================================
# BLOCK A: UNSUPERVISED LEARNING (K-MEANS & REGIME BOUNDARIES)
# ==============================================================================
def run_kmeans(df: pd.DataFrame):
    print("\n--- BLOCK A: K-MEANS & REGIME DETECTION ---")
    features_kmeans = ['debt_service_ratio', 'unemployment']
    df_cluster = df[features_kmeans].dropna()
    
    # Standardization
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_cluster)
    
    # 1. Dynamic Optimization (k=2 to 8) via Silhouette Score
    best_k, best_silhouette = 2, -1
    inertia_values = []
    
    for k in range(2, 9):
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans_temp.fit_predict(X_scaled)
        silhouette = silhouette_score(X_scaled, labels)
        inertia_values.append(kmeans_temp.inertia_)
        
        if silhouette > best_silhouette:
            best_silhouette = silhouette
            best_k = k
            
    print(f"[*] Optimization complete. Optimal K = {best_k} (Silhouette: {best_silhouette:.3f})")
    
    # 2. Final Estimation
    kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    df_cluster['Cluster'] = kmeans_final.fit_predict(X_scaled)
    centroids = scaler.inverse_transform(kmeans_final.cluster_centers_)
    
    # Centroid-based naming
    cluster_names = {}
    for c in range(best_k):
        effort, unemp = centroids[c, 0], centroids[c, 1]
        if unemp < 7.5:
            cluster_names[c] = 'Low Debt Burden & Low Unemployment' if effort < 12.5 else 'High Debt Burden & Low Unemployment'
        else:
            cluster_names[c] = 'High Debt Burden & High Unemployment'
            
    # 3. Export Statistics
    regime_limits = df_cluster.groupby('Cluster').agg({
        'debt_service_ratio': ['min', 'max'], 'unemployment': ['min', 'max']
    }).round(2)
    regime_limits.index = regime_limits.index.map(lambda x: cluster_names.get(x, f'Cluster {x}'))
    
    csv_filename = '04A_kmeans_regime_statistics.csv'
    regime_limits.to_csv(csv_filename, encoding='utf-8')
    print(f"[*] Statistics exported to '{csv_filename}'")
    
    # 4. K-Means Scatter Plot
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    fig, ax = plt.subplots(figsize=(9, 6))
    
    for c in range(best_k):
        subset = df_cluster[df_cluster['Cluster'] == c]
        ax.scatter(subset['debt_service_ratio'], subset['unemployment'], color=colors[c], 
                   alpha=0.75, s=60, edgecolors='k', linewidths=0.4, label=cluster_names[c])
        
    ax.scatter(centroids[:, 0], centroids[:, 1], color='black', marker='X', s=180, zorder=5, label='Centroids')
    
    ax.set_xlabel('Household Debt Service Ratio (%)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Unemployment Rate (%)', fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.legend(frameon=False, loc='best', fontsize=9)
    plt.tight_layout()
    plt.savefig('04A_kmeans_scatter.png', dpi=300, bbox_inches='tight')
    plt.close()

# ==============================================================================
# BLOCK B: SUPERVISED LEARNING (XGBOOST & SHAP VALUES)
# ==============================================================================
def run_xgboost_shap(df: pd.DataFrame):
    print("\n--- BLOCK B: XGBOOST & SHAP VALUES ---")
    TARGET = 'delinq_credit_cards'
    FEATURES = ['unemployment', 'fed_funds', 'spread_10y3m', 'debt_service_ratio', 'sloos_tightening']
    WINNING_LAG = 1
    PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)

    # Feature engineering (lags)
    X = pd.DataFrame(index=df.index)
    for col in FEATURES:
        X[f'{col}_lag{WINNING_LAG}'] = df[col].shift(WINNING_LAG)
        
    valid_rows = pd.concat([X, df[TARGET]], axis=1).dropna().index
    X_final, y_final = X.loc[valid_rows], df.loc[valid_rows, TARGET]
    
    print(f"[*] Training XGBoost (Lag {WINNING_LAG})...")
    final_model = xgb.XGBRegressor(**PARAMS).fit(X_final, y_final)
    
    print("[*] Extracting SHAP values...")
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_final)
    
    # 1. Global Importance Chart (Mean Absolute SHAP)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    clean_names = [col.replace(f'_lag{WINNING_LAG}', '').replace('_', ' ').title() for col in X_final.columns]
    
    df_plot = pd.DataFrame({'Feature': clean_names, 'Importance': mean_abs_shap}).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    bars = ax.barh(df_plot['Feature'], df_plot['Importance'], color='#d62728', height=0.6)
    
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.005, bar.get_y() + bar.get_height()/2, f'{width:.3f}', va='center', fontweight='bold', fontsize=10)
        
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, color='gray')
    ax.set_xlabel('Mean Impact on Delinquency Risk (|SHAP Value|)', fontsize=12, fontweight='bold', labelpad=15)
    
    ax.legend(handles=[Patch(facecolor='#d62728', label='Global Importance Magnitude')], loc='lower right', frameon=False)
    plt.tight_layout()
    plt.savefig('04B_shap_executive_panel.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Historical Crises Mapping on the Main Variable
    break_var = f'debt_service_ratio_lag{WINNING_LAG}'
    idx_effort = list(X_final.columns).index(break_var)
    
    plt.figure(figsize=(10, 7))
    plt.scatter(X_final[break_var], shap_values[:, idx_effort], color='#1f77b4', alpha=0.7, s=50, label='Historical Quarters')
    
    milestones = {
        '2008-10-01': 'Subprime Crisis (2008)',
        '2020-04-01': 'Pandemic Shock (2020)',
        '2022-07-01': 'Inflationary Crisis (2022)'
    }
    
    for date_str, label in milestones.items():
        ts_date = pd.to_datetime(date_str)
        if ts_date in X_final.index:
            loc_pos = np.where(X_final.index == ts_date)[0][0]
            x_val, y_val = float(X_final.loc[ts_date, break_var]), float(shap_values[loc_pos, idx_effort])
            plt.scatter(x_val, y_val, color='red', s=100, zorder=5)
            plt.annotate(label, (x_val, y_val), textcoords="offset points", xytext=(15, -5), fontweight='bold', fontsize=9,
                         bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5), arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0"))

    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.xlabel('Household Debt Service Ratio in the Previous Quarter (%)', fontsize=11, fontweight='bold')
    plt.ylabel('SHAP Impact (Driving Force)', fontsize=11, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig('04B_shap_historical_crises.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("[*] SHAP interpretability charts exported successfully.")

if __name__ == "__main__":
    run_kmeans(df_model)
    run_xgboost_shap(df_model)
    print("\n[SUCCESS] Complete Machine Learning (Appendix) pipeline finished.")